<a href="https://colab.research.google.com/github/ImaginationX4/Path_to_MARL/blob/master/MCST.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [55]:
import math
import random
import time

class TicTacToe:
    def __init__(self):
        # 初始化 9 个空格的棋盘
        self.board = [' ' for _ in range(9)]
        self.current_winner = None  # 跟踪当前赢家

    def print_board(self):
        # 打印棋盘
        for row in [self.board[i*3:(i+1)*3] for i in range(3)]:
            print('| ' + ' | '.join(row) + ' |')

    def available_moves(self):
        # 返回所有可用的移动（空格的索引）
        return [i for i, spot in enumerate(self.board) if spot == ' ']

    def empty_squares(self):
        # 检查棋盘上是否还有空格
        return ' ' in self.board

    def num_empty_squares(self):
        # 返回空格的数量
        return len(self.available_moves())

    def make_move(self, square, letter):
        # 如果有效，则进行移动并检查是否获胜
        if self.board[square] == ' ':
            self.board[square] = letter
            if self.winner(square, letter):
                self.current_winner = letter
            return True
        return False

    def winner(self, square, letter):
        # 检查是否有获胜者
        # 检查行
        row_ind = square // 3
        row = self.board[row_ind*3:(row_ind+1)*3]
        if all([s == letter for s in row]):
            return True
        # 检查列
        col_ind = square % 3
        column = [self.board[col_ind+i*3] for i in range(3)]
        if all([s == letter for s in column]):
            return True
        # 检查对角线
        if square % 2 == 0:
            diagonal1 = [self.board[i] for i in [0, 4, 8]]
            if all([s == letter for s in diagonal1]):
                return True
            diagonal2 = [self.board[i] for i in [2, 4, 6]]
            if all([s == letter for s in diagonal2]):
                return True
        return False

class Node:
    def __init__(self, state, parent=None, action=None):
        self.state = state  # 游戏状态
        self.parent = parent  # 父节点
        self.action = action  # 导致这个状态的动作
        self.children = []
          # 子节点
        self.p_children = []
        self.visits = 0  # 访问次数
        self.value = 0  # 节点的值


def MonteCarloTreeSearch(state, num_iterations=10000):

  if state.num_empty_squares() > 5:
      num_iterations = 1000  # 开局时少迭代
  else:
      num_iterations = 5000  # 终局时多迭代

  root = Node(state)
  for place in state.available_moves():
      action = place
      new_state = TicTacToe()
      new_state.board =     root.state.board.copy()
      new_state.make_move(action, 'X' if root.state.num_empty_squares() % 2 == 1 else 'O')
      child = Node(new_state, parent=root, action=action)
      root.children.append(child)


  for _ in range(num_iterations):

      leaf_node = select(root)
      result = simulate(leaf_node.state)
      backpropagate(leaf_node, result)



  return best_action(root)

def select(node):

  # 选择一个节点来扩展
  while not is_terminal(node.state) and node.children:
      node = best_uct(node)
  if not is_terminal(node.state):
      return expand(node)
  return node

def expand(node):

  actions = node.state.available_moves()
  action = random.choice(actions)

  new_state = TicTacToe()
  new_state.board = node.state.board.copy()
  new_state.make_move(action, 'X' if node.state.num_empty_squares() % 2 == 1 else 'O')
  child = Node(new_state, parent=node, action=action)
  node.children.append(child)

  return child



def simulate(state):
    # 模拟一次游戏，用启发式策略代替纯随机
    current_state = TicTacToe()
    current_state.board = state.board.copy()
    current_state.current_winner = state.current_winner

    while not is_terminal(current_state):
        # 优先选择中心点或角落，避免随机
        actions = current_state.available_moves()
        if 4 in actions:
            action = 4
        else:
            corners = [i for i in [0, 2, 6, 8] if i in actions]
            if corners:
                action = random.choice(corners)
            else:
                action = random.choice(actions)
        current_state.make_move(action, 'X' if current_state.num_empty_squares() % 2 == 1 else 'O')

    if current_state.current_winner == 'X':
        return 1
    elif current_state.current_winner == 'O':
        return -1
    else:
        return 0


def backpropagate(node, result):
    # 反向传播结果
    while node:

      node.visits += 1
      node.value += result
      node = node.parent

def best_uct(node):


  # 选择 UCT 值最高的子节点
  return max(node.children, key=lambda child: uct_value(child))

def uct_value(node, exploration_weight=1.01):
    # 计算节点的 UCT 值
    if node.visits == 0:
        return float('inf')  # 如果子节点还没有被访问过，优先选择它
    exploitation = node.value / node.visits  # 平均回报值
    exploration = exploration_weight * math.sqrt(math.log(node.parent.visits) / node.visits)
    return exploitation + exploration

def best_action(node):
    # 选择访问次数最多的动作
    return best_uct(node).action

def is_terminal(state):
    # 检查游戏是否结束
    return state.current_winner is not None or not state.empty_squares()

# 主游戏循环
def play_game():
    game = TicTacToe()
    while not is_terminal(game):
        game.print_board()
        if game.num_empty_squares() % 2 == 1:
            # AI的回合
            action = MonteCarloTreeSearch(game)
            game.make_move(action, 'X')
            print(f"AI (X) 选择了位置 {action}")
        else:
            # 玩家的回合
            valid_move = False
            while not valid_move:
                try:
                    action = int(input("请输入您的移动 (1-9): "))
                    valid_move = game.make_move(action-1, 'O')
                    if not valid_move:
                        print("无效的移动，请重试。")
                except ValueError:
                    print("请输入一个有效的数字。")

    game.print_board()
    if game.current_winner:
        print(f"获胜者是 {game.current_winner}!")
    else:
        print("平局!")

if __name__ == "__main__":
    play_game()

|   |   |   |
|   |   |   |
|   |   |   |
AI (X) 选择了位置 8
|   |   |   |
|   |   |   |
|   |   | X |
请输入您的移动 (1-9): 5
|   |   |   |
|   | O |   |
|   |   | X |
AI (X) 选择了位置 1
|   | X |   |
|   | O |   |
|   |   | X |
请输入您的移动 (1-9): 3
|   | X | O |
|   | O |   |
|   |   | X |
AI (X) 选择了位置 6
|   | X | O |
|   | O |   |
| X |   | X |
请输入您的移动 (1-9): 1
| O | X | O |
|   | O |   |
| X |   | X |
AI (X) 选择了位置 3
| O | X | O |
| X | O |   |
| X |   | X |
请输入您的移动 (1-9): 6
| O | X | O |
| X | O | O |
| X |   | X |
AI (X) 选择了位置 7
| O | X | O |
| X | O | O |
| X | X | X |
获胜者是 X!
